## Homework 2 Additional Boosting

In [34]:
# Required imports
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

# LightGBM
from lightgbm import LGBMClassifier

# CatBoost
from catboost import CatBoostClassifier

In [35]:
train_df = pd.read_csv("playground-series-s6e4/train.csv")
test_df = pd.read_csv("playground-series-s6e4/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Set target column explicitly for this competition
target_col = "Irrigation_Need"
id_col = "id" if "id" in train_df.columns else None

print("Target column:", target_col)
print("ID column:", id_col)
print("Target values:", train_df[target_col].unique())

Train shape: (630000, 21)
Test shape: (270000, 20)
Target column: Irrigation_Need
ID column: id
Target values: <StringArray>
['Low', 'Medium', 'High']
Length: 3, dtype: str


In [36]:
feature_cols = [c for c in train_df.columns if c not in [target_col, id_col]] if id_col else [c for c in train_df.columns if c != target_col]

X = train_df[feature_cols].copy()
y = train_df[target_col].copy()
X_test = test_df[feature_cols].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("Target in X_train?", target_col in X_train.columns)
print("y_train dtype:", y_train.dtype)
print("Unique target values:", y_train.unique())

# Encode target labels for models that require numeric classes
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_valid_enc = label_encoder.transform(y_valid)

print("Encoded classes:", list(label_encoder.classes_))

X_train: (504000, 19)
X_valid: (126000, 19)
Target in X_train? False
y_train dtype: str
Unique target values: <StringArray>
['Medium', 'Low', 'High']
Length: 3, dtype: str
Encoded classes: ['High', 'Low', 'Medium']


In [37]:
# LightGBM baseline
# One-hot encode categoricals for a simple baseline
X_train_lgb = pd.get_dummies(X_train, drop_first=False)
X_valid_lgb = pd.get_dummies(X_valid, drop_first=False)
X_test_lgb = pd.get_dummies(X_test, drop_first=False)

# Align columns after one-hot encoding
X_train_lgb, X_valid_lgb = X_train_lgb.align(X_valid_lgb, join="left", axis=1, fill_value=0)
X_train_lgb, X_test_lgb = X_train_lgb.align(X_test_lgb, join="left", axis=1, fill_value=0)

lgb_model = LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

lgb_model.fit(X_train_lgb, y_train_enc)

lgb_valid_preds = lgb_model.predict(X_valid_lgb)

lgb_acc = accuracy_score(y_valid_enc, lgb_valid_preds)
lgb_bal_acc = balanced_accuracy_score(y_valid_enc, lgb_valid_preds)

print(f"LightGBM Accuracy: {lgb_acc:.6f}")
print(f"LightGBM Balanced Accuracy: {lgb_bal_acc:.6f}")
print(classification_report(y_valid_enc, lgb_valid_preds, target_names=label_encoder.classes_))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004567 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2726
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 43
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
LightGBM Accuracy: 0.985294
LightGBM Balanced Accuracy: 0.964793
              precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



In [38]:
# CatBoost baseline
# CatBoost can directly use categorical columns
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
cat_feature_indices = [X_train.columns.get_loc(col) for col in cat_features]

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=0,
)

cat_model.fit(
    X_train,
    y_train_enc,
    cat_features=cat_feature_indices,
    eval_set=(X_valid, y_valid_enc),
    use_best_model=True,
)

cat_valid_preds = cat_model.predict(X_valid)
cat_valid_preds = cat_valid_preds.flatten().astype(int)

cat_acc = accuracy_score(y_valid_enc, cat_valid_preds)
cat_bal_acc = balanced_accuracy_score(y_valid_enc, cat_valid_preds)

print(f"CatBoost Accuracy: {cat_acc:.6f}")
print(f"CatBoost Balanced Accuracy: {cat_bal_acc:.6f}")
print(classification_report(y_valid_enc, cat_valid_preds, target_names=label_encoder.classes_))

/var/folders/kk/czyhbh5961ggwtk14x632d900000gq/T/ipykernel_94825/1383441612.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


CatBoost Accuracy: 0.985071
CatBoost Balanced Accuracy: 0.963823
              precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



In [39]:
param_grid_1 = {
    "learning_rate": 0.05,
    "n_estimators": 500,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

param_grid_2 = {
    "learning_rate": 0.03,
    "n_estimators": 300,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 20,
    "random_state": 42
}

param_grid_3 = {
    "learning_rate": 0.02,
    "n_estimators": 500,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "random_state": 42
}

param_grids = {
    "LightGBM_Model_1": param_grid_1,
    "LightGBM_Model_2": param_grid_2,
    "LightGBM_Model_3": param_grid_3
}

In [40]:
lgb_results = []
lgb_models = {}

for model_name, params in param_grids.items():
    model = LGBMClassifier(
        objective="multiclass",
        num_class=len(label_encoder.classes_),
        **params
    )
    
    model.fit(X_train_lgb, y_train_enc)
    preds = model.predict(X_valid_lgb)
    
    acc = accuracy_score(y_valid_enc, preds)
    bal_acc = balanced_accuracy_score(y_valid_enc, preds)
    
    lgb_results.append({
        "Model": model_name,
        "learning_rate": params.get("learning_rate"),
        "n_estimators": params.get("n_estimators"),
        "num_leaves": params.get("num_leaves"),
        "max_depth": params.get("max_depth"),
        "min_child_samples": params.get("min_child_samples"),
        "subsample": params.get("subsample", 1.0),
        "colsample_bytree": params.get("colsample_bytree", 1.0),
        "Validation Accuracy": acc,
        "Validation Balanced Accuracy": bal_acc
    })
    
    lgb_models[model_name] = model
    
    print(f"\n{model_name}")
    print("-" * 50)
    print(f"Accuracy: {acc:.6f}")
    print(f"Balanced Accuracy: {bal_acc:.6f}")
    print(classification_report(y_valid_enc, preds, target_names=label_encoder.classes_))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005346 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2726
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 43
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948

LightGBM_Model_1
--------------------------------------------------
Accuracy: 0.985294
Balanced Accuracy: 0.964793
              precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.9

In [41]:
lgb_results_df = pd.DataFrame(lgb_results).sort_values(
    by="Validation Balanced Accuracy",
    ascending=False
).reset_index(drop=True)

lgb_results_df

,Model,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,colsample_bytree,Validation Accuracy,Validation Balanced Accuracy
0,LightGBM_Model_1,0.05,500,31,-1,20,0.8,0.8,0.985294,0.964793
1,LightGBM_Model_2,0.03,300,63,-1,20,1.0,1.0,0.985175,0.964389
2,LightGBM_Model_3,0.02,500,31,-1,20,1.0,1.0,0.985032,0.963809


In [42]:
best_lgb_test_preds = best_lgb_model.predict(X_test_lgb)
best_lgb_test_labels = label_encoder.inverse_transform(best_lgb_test_preds.astype(int))

submission_lgb = pd.DataFrame({
    "id": test_df["id"],
    "Irrigation_Need": best_lgb_test_labels
})

submission_lgb.to_csv("Output/best_lightgbm_submission.csv", index=False)
submission_lgb.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [43]:
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
cat_feature_indices = [X_train.columns.get_loc(col) for col in cat_features]

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=0
)

cat_model.fit(
    X_train,
    y_train_enc,
    cat_features=cat_feature_indices,
    eval_set=(X_valid, y_valid_enc),
    use_best_model=True
)

cat_valid_preds = cat_model.predict(X_valid)
cat_valid_preds = cat_valid_preds.flatten().astype(int)

cat_acc = accuracy_score(y_valid_enc, cat_valid_preds)
cat_bal_acc = balanced_accuracy_score(y_valid_enc, cat_valid_preds)

print(f"CatBoost Accuracy: {cat_acc:.6f}")
print(f"CatBoost Balanced Accuracy: {cat_bal_acc:.6f}")
print(classification_report(y_valid_enc, cat_valid_preds, target_names=label_encoder.classes_))

/var/folders/kk/czyhbh5961ggwtk14x632d900000gq/T/ipykernel_94825/418425973.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


CatBoost Accuracy: 0.985071
CatBoost Balanced Accuracy: 0.963823
              precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



In [44]:
cat_test_preds = cat_model.predict(X_test)
cat_test_preds = cat_test_preds.flatten().astype(int)
cat_test_labels = label_encoder.inverse_transform(cat_test_preds)

submission_cat = pd.DataFrame({
    "id": test_df["id"],
    "Irrigation_Need": cat_test_labels
})

submission_cat.to_csv("Output/catboost_submission.csv", index=False)
submission_cat.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [45]:
# New LightGBM model with updated hyperparameters

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

param_grid_new = {
    "learning_rate": 0.05,
    "n_estimators": 500,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

lgb_model_new = LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),
    **param_grid_new
)

# Train
lgb_model_new.fit(X_train_lgb, y_train_enc)

# Predict
lgb_valid_preds_new = lgb_model_new.predict(X_valid_lgb)

# Evaluate
acc_new = accuracy_score(y_valid_enc, lgb_valid_preds_new)
bal_acc_new = balanced_accuracy_score(y_valid_enc, lgb_valid_preds_new)

print("New LightGBM Model (num_leaves=63)")
print("-" * 50)
print(f"Accuracy: {acc_new:.6f}")
print(f"Balanced Accuracy: {bal_acc_new:.6f}")
print(classification_report(y_valid_enc, lgb_valid_preds_new, target_names=label_encoder.classes_))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004010 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2726
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 43
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
New LightGBM Model (num_leaves=63)
--------------------------------------------------
Accuracy: 0.985175
Balanced Accuracy: 0.964040
              precision    recall  f1-score   support

        High       0.96      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99  

In [46]:
test_preds_new = lgb_model_new.predict(X_test_lgb)
test_labels_new = label_encoder.inverse_transform(test_preds_new.astype(int))

submission_new = pd.DataFrame({
    "id": test_df["id"],
    "Irrigation_Need": test_labels_new
})

submission_new.to_csv("Output/lightgbm_model_num_leaves_63.csv", index=False)
submission_new.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [47]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import balanced_accuracy_score

def objective(trial):
    
    params = {
        "objective": "multiclass",
        "num_class": len(label_encoder.classes_),
        
        # Hyperparameters to tune
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        
        "random_state": 42
    }

    model = LGBMClassifier(**params)
    
    model.fit(X_train_lgb, y_train_enc)
    
    preds = model.predict(X_valid_lgb)
    
    score = balanced_accuracy_score(y_valid_enc, preds)
    
    return score

In [48]:
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=20)  # you can increase to 50–100 later

[I 2026-04-20 23:02:08,686] A new study created in memory with name: no-name-37f7274e-3269-4cb9-84b0-27d46f107d91


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006816 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2726
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 43
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948


[I 2026-04-20 23:02:17,159] Trial 0 finished with value: 0.9631868911257229 and parameters: {'n_estimators': 140, 'learning_rate': 0.02049879442830399, 'num_leaves': 45, 'min_child_samples': 15, 'subsample': 0.8461463871316688, 'colsample_bytree': 0.9969396983964576}. Best is trial 0 with value: 0.9631868911257229.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004741 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2726
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 43
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948


[I 2026-04-20 23:02:30,201] Trial 1 finished with value: 0.964907664570414 and parameters: {'n_estimators': 171, 'learning_rate': 0.08284418009295189, 'num_leaves': 69, 'min_child_samples': 41, 'subsample': 0.9547224239426256, 'colsample_bytree': 0.8780453474569889}. Best is trial 1 with value: 0.964907664570414.


...

[I 2026-04-20 23:08:12,296] Trial 17 finished with value: 0.9642445762261764 and parameters: {'n_estimators': 162, 'learning_rate': 0.04443094048992471, 'num_leaves': 35, 'min_child_samples': 42, 'subsample': 0.8532919302855156, 'colsample_bytree': 0.9195456633063797}. Best is trial 8 with value: 0.9650343543949932.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005510 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2726
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 43
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948


[I 2026-04-20 23:08:28,160] Trial 18 finished with value: 0.9645274364477027 and parameters: {'n_estimators': 235, 'learning_rate': 0.07918874442157021, 'num_leaves': 61, 'min_child_samples': 37, 'subsample': 0.9468154690757917, 'colsample_bytree': 0.8663804685806835}. Best is trial 8 with value: 0.9650343543949932.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004962 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2726
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 43
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948


[I 2026-04-20 23:08:55,804] Trial 19 finished with value: 0.9640176267195151 and parameters: {'n_estimators': 352, 'learning_rate': 0.04821813582593601, 'num_leaves': 72, 'min_child_samples': 29, 'subsample': 0.7510389135102057, 'colsample_bytree': 0.8202754212117036}. Best is trial 8 with value: 0.9650343543949932.


In [49]:
print("Best trial score:", study.best_value)
print("Best parameters:")
print(study.best_params)

Best trial score: 0.9650343543949932
Best parameters:
{'n_estimators': 283, 'learning_rate': 0.07929405970655987, 'num_leaves': 40, 'min_child_samples': 20, 'subsample': 0.7510753080943224, 'colsample_bytree': 0.8862841860670488}


In [50]:
best_params = study.best_params

best_model_optuna = LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),
    random_state=42,
    **best_params
)

best_model_optuna.fit(X_train_lgb, y_train_enc)

optuna_preds = best_model_optuna.predict(X_valid_lgb)

from sklearn.metrics import accuracy_score

print("Optuna Accuracy:", accuracy_score(y_valid_enc, optuna_preds))
print("Optuna Balanced Accuracy:", balanced_accuracy_score(y_valid_enc, optuna_preds))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004563 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2726
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 43
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
Optuna Accuracy: 0.9852222222222222
Optuna Balanced Accuracy: 0.9650343543949932


In [51]:
test_preds_optuna = best_model_optuna.predict(X_test_lgb)
test_labels_optuna = label_encoder.inverse_transform(test_preds_optuna.astype(int))

submission_optuna = pd.DataFrame({
    "id": test_df["id"],
    "Irrigation_Need": test_labels_optuna
})

submission_optuna.to_csv("Output/optuna_lightgbm_submission.csv", index=False)
submission_optuna.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [ ]:
# Cross-validated Balanced Accuracy for all requested models
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Full training data (no holdout split) for CV
X_full = train_df[feature_cols].copy()
y_full = label_encoder.transform(train_df[target_col])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---------- 1) Baseline CatBoost ----------
cat_features_cv = X_full.select_dtypes(include=["object", "category", "string"]).columns.tolist()
cat_feature_indices_cv = [X_full.columns.get_loc(c) for c in cat_features_cv]

cat_base_cv = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="MultiClass",
    random_seed=42,
    verbose=0,
)

# Manual CV loop (compatible with newer sklearn where cross_val_score fit_params is removed)
cat_cv_scores = []
for fold, (train_idx, valid_idx) in enumerate(skf.split(X_full, y_full), start=1):
    X_tr = X_full.iloc[train_idx]
    X_va = X_full.iloc[valid_idx]
    y_tr = y_full[train_idx]
    y_va = y_full[valid_idx]

    cat_fold_model = CatBoostClassifier(**cat_base_cv.get_params())
    cat_fold_model.fit(X_tr, y_tr, cat_features=cat_feature_indices_cv)

    cat_preds = cat_fold_model.predict(X_va).flatten().astype(int)
    fold_bal_acc = balanced_accuracy_score(y_va, cat_preds)
    cat_cv_scores.append(fold_bal_acc)

cat_cv_scores = np.array(cat_cv_scores)

# ---------- 2) Best LightGBM ----------
# Reuse your selected best model if available; otherwise use the top row from lgb_results_df.
if "best_lgb_model" in globals():
    best_lgb_params = best_lgb_model.get_params()
elif "lgb_results_df" in globals():
    top_row = lgb_results_df.iloc[0]
    best_lgb_params = {
        "learning_rate": float(top_row["learning_rate"]),
        "n_estimators": int(top_row["n_estimators"]),
        "num_leaves": int(top_row["num_leaves"]),
        "max_depth": int(top_row["max_depth"]),
        "min_child_samples": int(top_row["min_child_samples"]),
        "subsample": float(top_row["subsample"]),
        "colsample_bytree": float(top_row["colsample_bytree"]),
        "objective": "multiclass",
        "num_class": len(label_encoder.classes_),
        "random_state": 42,
    }
else:
    best_lgb_params = {
        "learning_rate": 0.05,
        "n_estimators": 500,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "objective": "multiclass",
        "num_class": len(label_encoder.classes_),
        "random_state": 42,
    }

best_lgb_cv = LGBMClassifier(**best_lgb_params)

lgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            X_full.select_dtypes(include=["object", "category", "string"]).columns.tolist(),
        )
    ],
    remainder="passthrough",
)

best_lgb_pipeline = Pipeline([
    ("prep", lgb_preprocessor),
    ("model", best_lgb_cv),
])

best_lgb_cv_scores = cross_val_score(
    best_lgb_pipeline,
    X_full,
    y_full,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1,
)

# ---------- 3) Modified LightGBM ----------
if "lgb_model_new" in globals():
    mod_lgb_params = lgb_model_new.get_params()
else:
    mod_lgb_params = {
        "objective": "multiclass",
        "num_class": len(label_encoder.classes_),
        "n_estimators": 800,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "min_child_samples": 10,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "random_state": 42,
    }

mod_lgb_cv = LGBMClassifier(**mod_lgb_params)
mod_lgb_pipeline = Pipeline([
    ("prep", lgb_preprocessor),
    ("model", mod_lgb_cv),
])

mod_lgb_cv_scores = cross_val_score(
    mod_lgb_pipeline,
    X_full,
    y_full,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1,
)

# ---------- 4) Optuna LightGBM ----------
if "best_model_optuna" in globals():
    optuna_lgb_params = best_model_optuna.get_params()
elif "best_params" in globals():
    optuna_lgb_params = {
        "objective": "multiclass",
        "num_class": len(label_encoder.classes_),
        "random_state": 42,
        **best_params,
    }
else:
    optuna_lgb_params = {
        "objective": "multiclass",
        "num_class": len(label_encoder.classes_),
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "random_state": 42,
    }

optuna_lgb_cv = LGBMClassifier(**optuna_lgb_params)
optuna_lgb_pipeline = Pipeline([
    ("prep", lgb_preprocessor),
    ("model", optuna_lgb_cv),
])

optuna_lgb_cv_scores = cross_val_score(
    optuna_lgb_pipeline,
    X_full,
    y_full,
    cv=skf,
    scoring="balanced_accuracy",
    n_jobs=-1,
)

# ---------- Summary ----------
cv_summary = pd.DataFrame(
    {
        "Model": [
            "CatBoost Baseline",
            "LightGBM Best",
            "LightGBM Modified",
            "LightGBM Optuna",
        ],
        "CV Mean Balanced Accuracy": [
            cat_cv_scores.mean(),
            best_lgb_cv_scores.mean(),
            mod_lgb_cv_scores.mean(),
            optuna_lgb_cv_scores.mean(),
        ],
        "CV Std": [
            cat_cv_scores.std(),
            best_lgb_cv_scores.std(),
            mod_lgb_cv_scores.std(),
            optuna_lgb_cv_scores.std(),
        ],
    }
).sort_values("CV Mean Balanced Accuracy", ascending=False)

cv_summary